<a href="https://colab.research.google.com/github/DLHolmes4/Coding-demo/blob/main/portablecoreV1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# VLCP — FRESH NOTEBOOK PORTABILITY TEST
#
# This notebook has NO dependency on the development notebook.
#
# Loads:
#   Google Drive persistent VLCP package
#   FMCSA credential from Colab Secrets
#
# Then performs a live carrier verification.
# ============================================================

from google.colab import drive, userdata

import sys
import os


# ------------------------------------------------------------
# 1. MOUNT PERSISTENT STORAGE
# ------------------------------------------------------------

drive.mount(
    "/content/drive"
)


# ------------------------------------------------------------
# 2. LOCATE VLCP PACKAGE
# ------------------------------------------------------------

VLCP_PROJECT_ROOT = (
    "/content/drive/MyDrive/VLCP_PoC"
)

VLCP_PACKAGE = os.path.join(
    VLCP_PROJECT_ROOT,
    "vlcp"
)


if not os.path.isdir(VLCP_PACKAGE):

    raise FileNotFoundError(
        f"VLCP package not found: {VLCP_PACKAGE}"
    )


# ------------------------------------------------------------
# 3. MAKE PACKAGE IMPORTABLE
#
# We add the DIRECTORY CONTAINING vlcp,
# not the vlcp directory itself.
# ------------------------------------------------------------

if VLCP_PROJECT_ROOT not in sys.path:

    sys.path.insert(
        0,
        VLCP_PROJECT_ROOT
    )


# ------------------------------------------------------------
# 4. IMPORT VLCP
# ------------------------------------------------------------

import vlcp
import vlcp.core as core


# ------------------------------------------------------------
# 5. LOAD CREDENTIAL FROM COLAB SECRETS
#
# Secret value is never printed.
# ------------------------------------------------------------

FMCSA_WEBKEY = userdata.get(
    "FMCSA_WEBKEY"
)


if not FMCSA_WEBKEY:

    raise RuntimeError(
        "FMCSA_WEBKEY was not found in Colab Secrets."
    )


# Inject credential into package runtime.
core.FMCSA_WEBKEY = FMCSA_WEBKEY


# ------------------------------------------------------------
# 6. DYNAMIC TEST INPUT
#
# Change this to another valid USDOT later without modifying
# VLCP business logic.
# ------------------------------------------------------------

TEST_USDOT = "951224"


# ------------------------------------------------------------
# 7. LIVE VLCP VERIFICATION
# ------------------------------------------------------------

result = vlcp.verify_carrier(
    TEST_USDOT
)


identity = (
    result.get("identity")
    or {}
)


# ------------------------------------------------------------
# 8. REPORT
# ------------------------------------------------------------

print("=" * 78)
print("VLCP — FRESH NOTEBOOK PORTABILITY TEST")
print("=" * 78)

print(
    "Package loaded from:",
    vlcp.__file__
)

print(
    "Core loaded from:",
    core.__file__
)

print()

print(
    "Carrier:",
    identity.get("display_name")
)

print(
    "USDOT:",
    result.get("usdot")
)

print(
    "Evidence status:",
    result.get("verification_status")
)


print("\nMODULE STATUS")
print("-" * 78)


for module, status in (
    result.get(
        "module_status",
        {}
    )
).items():

    print(
        f"{module:<22} {status}"
    )


print("\nERRORS")
print("-" * 78)


errors = result.get(
    "errors",
    []
)


if errors:

    for error in errors:
        print("•", error)

else:

    print("None")


# ------------------------------------------------------------
# 9. PORTABILITY ASSERTIONS
# ------------------------------------------------------------

assert (
    "/content/drive/MyDrive/VLCP_PoC/vlcp"
    in vlcp.__file__
), (
    "VLCP was not imported from the persistent Drive package."
)


assert (
    result.get("usdot")
    == TEST_USDOT
)


assert (
    identity.get("display_name")
    == "US SERVICES LLC"
)


assert (
    result.get("verification_status")
    == "EVIDENCE_RETRIEVED"
)


expected_modules = {

    "identity":
        "FOUND",

    "authority":
        "RETRIEVED",

    "insurance":
        "RETRIEVED",

    "insurance_lifecycle":
        "RETRIEVED",
}


for module, expected in (
    expected_modules.items()
):

    actual = (
        result
        .get("module_status", {})
        .get(module)
    )

    assert actual == expected, (
        f"{module}: "
        f"expected {expected}, "
        f"got {actual}"
    )


assert not errors, (
    f"Unexpected errors: {errors}"
)


print("\n" + "=" * 78)

print(
    "✓ VLCP PORTABLE CORE v1 PASSED "
    "FRESH-NOTEBOOK TEST"
)

print("=" * 78)

Mounted at /content/drive
VLCP — FRESH NOTEBOOK PORTABILITY TEST
Package loaded from: /content/drive/MyDrive/VLCP_PoC/vlcp/__init__.py
Core loaded from: /content/drive/MyDrive/VLCP_PoC/vlcp/core.py

Carrier: US SERVICES LLC
USDOT: 951224
Evidence status: EVIDENCE_RETRIEVED

MODULE STATUS
------------------------------------------------------------------------------
identity               FOUND
authority              RETRIEVED
insurance              RETRIEVED
insurance_lifecycle    RETRIEVED

ERRORS
------------------------------------------------------------------------------
None

✓ VLCP PORTABLE CORE v1 PASSED FRESH-NOTEBOOK TEST


In [ ]:
# ============================================================
# VLCP — PACKAGE REPAIR 001
#
# Defect:
#   _socrata_query() requires SOCRATA_DOMAIN,
#   but the constant was omitted from the portable checkpoint.
#
# Repair:
#   Derive SOCRATA_DOMAIN from the already-verified FMCSA
#   Socrata URLs contained in the package and persist it
#   into core.py.
#
# This does NOT modify verification/business logic.
# ============================================================

import os
import re
import shutil
from urllib.parse import urlparse


CORE_FILE = (
    "/content/drive/MyDrive/"
    "VLCP_PoC/vlcp/core.py"
)

BACKUP_FILE = (
    "/content/drive/MyDrive/"
    "VLCP_PoC/vlcp/core_before_repair_001.py"
)


# ------------------------------------------------------------
# 1. VERIFY PACKAGE FILE
# ------------------------------------------------------------

if not os.path.isfile(CORE_FILE):

    raise FileNotFoundError(
        f"VLCP core not found: {CORE_FILE}"
    )


# ------------------------------------------------------------
# 2. READ CURRENT PACKAGE SOURCE
# ------------------------------------------------------------

with open(
    CORE_FILE,
    "r",
    encoding="utf-8"
) as f:

    source = f.read()


# ------------------------------------------------------------
# 3. MAKE BACKUP BEFORE MODIFYING PACKAGE
# ------------------------------------------------------------

if not os.path.exists(BACKUP_FILE):

    shutil.copy2(
        CORE_FILE,
        BACKUP_FILE
    )


# ------------------------------------------------------------
# 4. FIND EXISTING VERIFIED SOCRATA URL
# ------------------------------------------------------------

url_pattern = re.compile(
    r'''FMCSA_[A-Z_]+_URL\s*=\s*['"]([^'"]+)['"]'''
)

matches = url_pattern.findall(
    source
)


if not matches:

    raise RuntimeError(
        "Could not locate an existing FMCSA "
        "Socrata URL in core.py."
    )


# ------------------------------------------------------------
# 5. DERIVE DOMAIN
# ------------------------------------------------------------

parsed = urlparse(
    matches[0]
)


SOCRATA_DOMAIN_VALUE = (
    f"{parsed.scheme}://{parsed.netloc}"
)


if not parsed.scheme or not parsed.netloc:

    raise RuntimeError(
        "Could not derive Socrata domain "
        "from existing package URL."
    )


# ------------------------------------------------------------
# 6. CONFIRM ALL FMCSA SOCRATA URLS AGREE
# ------------------------------------------------------------

domains = {
    f"{urlparse(url).scheme}://"
    f"{urlparse(url).netloc}"
    for url in matches
}


if len(domains) != 1:

    raise RuntimeError(
        "FMCSA Socrata URLs do not share "
        f"one domain: {domains}"
    )


# ------------------------------------------------------------
# 7. CHECK WHETHER CONSTANT ALREADY EXISTS
# ------------------------------------------------------------

constant_definition = re.search(
    r"^SOCRATA_DOMAIN\s*=",
    source,
    flags=re.MULTILINE
)


if constant_definition:

    print(
        "SOCRATA_DOMAIN already exists "
        "in package source."
    )

else:

    # Insert immediately before the first
    # FMCSA configuration constant.

    marker = (
        "FMCSA_ACTPEND_INSURANCE_DATASET"
    )

    marker_position = source.find(
        marker
    )


    if marker_position == -1:

        raise RuntimeError(
            "Could not locate configuration "
            "insertion point."
        )


    line_start = source.rfind(
        "\n",
        0,
        marker_position
    ) + 1


    addition = (
        f"SOCRATA_DOMAIN = "
        f"{SOCRATA_DOMAIN_VALUE!r}\n"
    )


    source = (
        source[:line_start]
        + addition
        + source[line_start:]
    )


    with open(
        CORE_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(source)


# ------------------------------------------------------------
# 8. COMPILE REPAIRED PACKAGE
# ------------------------------------------------------------

with open(
    CORE_FILE,
    "r",
    encoding="utf-8"
) as f:

    repaired_source = f.read()


compile(
    repaired_source,
    CORE_FILE,
    "exec"
)


# ------------------------------------------------------------
# 9. VERIFY CONSTANT IS ACTUALLY PRESENT
# ------------------------------------------------------------

assert re.search(
    r"^SOCRATA_DOMAIN\s*=",
    repaired_source,
    flags=re.MULTILINE
), "SOCRATA_DOMAIN was not persisted."


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print("=" * 78)
print("VLCP PACKAGE REPAIR 001")
print("=" * 78)

print(
    "Derived Socrata domain:",
    SOCRATA_DOMAIN_VALUE
)

print(
    "Package:",
    CORE_FILE
)

print(
    "Backup:",
    BACKUP_FILE
)

print(
    "Compile test: PASS"
)

print()

print(
    "✓ SOCRATA_DOMAIN PERSISTED "
    "IN VLCP PACKAGE"
)

print("=" * 78)

VLCP PACKAGE REPAIR 001
Derived Socrata domain: https://data.transportation.gov
Package: /content/drive/MyDrive/VLCP_PoC/vlcp/core.py
Backup: /content/drive/MyDrive/VLCP_PoC/vlcp/core_before_repair_001.py
Compile test: PASS

✓ SOCRATA_DOMAIN PERSISTED IN VLCP PACKAGE


In [ ]:
# ============================================================
# VLCP — COMPLETE PACKAGE DEPENDENCY AUDIT v2
#
# Audits ALL functions now contained in persistent core.py,
# including internal helper functions.
#
# Goal:
# Detect unresolved package globals BEFORE another live test.
# ============================================================

import ast
import builtins
import os


CORE_FILE = (
    "/content/drive/MyDrive/"
    "VLCP_PoC/vlcp/core.py"
)


# ------------------------------------------------------------
# LOAD PACKAGE SOURCE
# ------------------------------------------------------------

with open(
    CORE_FILE,
    "r",
    encoding="utf-8"
) as f:

    source = f.read()


tree = ast.parse(source)


# ------------------------------------------------------------
# TOP-LEVEL DEFINITIONS
# ------------------------------------------------------------

top_level_names = set()


for node in tree.body:

    # Functions / classes
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
            ast.ClassDef
        )
    ):

        top_level_names.add(
            node.name
        )

    # Imports
    elif isinstance(
        node,
        ast.Import
    ):

        for alias in node.names:

            top_level_names.add(
                alias.asname
                or alias.name.split(".")[0]
            )

    elif isinstance(
        node,
        ast.ImportFrom
    ):

        for alias in node.names:

            top_level_names.add(
                alias.asname
                or alias.name
            )

    # Constants / assignments
    elif isinstance(
        node,
        (
            ast.Assign,
            ast.AnnAssign
        )
    ):

        targets = []

        if isinstance(
            node,
            ast.Assign
        ):

            targets = node.targets

        else:

            targets = [
                node.target
            ]

        for target in targets:

            if isinstance(
                target,
                ast.Name
            ):

                top_level_names.add(
                    target.id
                )


# ------------------------------------------------------------
# FUNCTION LOCAL-NAME ANALYSIS
# ------------------------------------------------------------

builtin_names = set(
    dir(builtins)
)

external_references = set()


class FunctionDependencyVisitor(
    ast.NodeVisitor
):

    def visit_FunctionDef(
        self,
        node
    ):

        local_names = set()


        # Parameters
        for arg in (
            list(node.args.posonlyargs)
            + list(node.args.args)
            + list(node.args.kwonlyargs)
        ):

            local_names.add(
                arg.arg
            )


        if node.args.vararg:

            local_names.add(
                node.args.vararg.arg
            )


        if node.args.kwarg:

            local_names.add(
                node.args.kwarg.arg
            )


        # Local assignments
        for child in ast.walk(node):

            if isinstance(
                child,
                ast.Name
            ):

                if isinstance(
                    child.ctx,
                    (
                        ast.Store,
                        ast.Param
                    )
                ):

                    local_names.add(
                        child.id
                    )


            # Exception aliases such as:
            # except Exception as exc
            if isinstance(
                child,
                ast.ExceptHandler
            ):

                if isinstance(
                    child.name,
                    str
                ):

                    local_names.add(
                        child.name
                    )


        # Inspect loaded names
        for child in ast.walk(node):

            if not isinstance(
                child,
                ast.Name
            ):

                continue


            if not isinstance(
                child.ctx,
                ast.Load
            ):

                continue


            name = child.id


            if name in local_names:
                continue


            if name in builtin_names:
                continue


            if name in top_level_names:
                continue


            external_references.add(
                name
            )


        # Do not recursively process this
        # function again through generic_visit.
        return


visitor = (
    FunctionDependencyVisitor()
)

visitor.visit(tree)


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print("=" * 78)
print("VLCP COMPLETE PACKAGE DEPENDENCY AUDIT v2")
print("=" * 78)

print(
    "Package:",
    CORE_FILE
)

print(
    "Top-level definitions:",
    len(top_level_names)
)

print(
    "Unresolved references:",
    len(external_references)
)


if external_references:

    print(
        "\nUNRESOLVED GLOBAL REFERENCES"
    )

    print("-" * 78)

    for name in sorted(
        external_references
    ):

        print(
            "✗",
            name
        )

else:

    print(
        "\n✓ NO UNRESOLVED GLOBAL "
        "REFERENCES DETECTED"
    )


print("\n" + "=" * 78)


if not external_references:

    print(
        "✓ VLCP PACKAGE STATIC "
        "DEPENDENCY AUDIT PASSED"
    )

else:

    print(
        "⚠ VLCP PACKAGE REQUIRES "
        "DEPENDENCY REPAIR"
    )


print("=" * 78)

VLCP COMPLETE PACKAGE DEPENDENCY AUDIT v2
Package: /content/drive/MyDrive/VLCP_PoC/vlcp/core.py
Top-level definitions: 52
Unresolved references: 1

UNRESOLVED GLOBAL REFERENCES
------------------------------------------------------------------------------
✗ _extract_socrata_rows

⚠ VLCP PACKAGE REQUIRES DEPENDENCY REPAIR


In [ ]:
# ============================================================
# VLCP — LOCATE VERIFIED _extract_socrata_rows SOURCE
#
# Search persistent VLCP artifacts for the exact helper
# implementation before making any repair.
#
# NO files are modified by this cell.
# ============================================================

import os
import re


VLCP_ROOT = (
    "/content/drive/MyDrive/VLCP_PoC"
)

TARGET = "_extract_socrata_rows"


# ------------------------------------------------------------
# FILE TYPES WORTH SEARCHING
# ------------------------------------------------------------

SEARCH_EXTENSIONS = {
    ".py",
    ".json",
    ".txt",
    ".ipynb",
}


matches = []


# ------------------------------------------------------------
# SEARCH ALL PERSISTENT VLCP ARTIFACTS
# ------------------------------------------------------------

for root, dirs, files in os.walk(
    VLCP_ROOT
):

    # Skip Python cache files.
    dirs[:] = [
        d for d in dirs
        if d != "__pycache__"
    ]

    for filename in files:

        path = os.path.join(
            root,
            filename
        )

        extension = os.path.splitext(
            filename
        )[1].lower()


        if extension not in SEARCH_EXTENSIONS:
            continue


        try:

            with open(
                path,
                "r",
                encoding="utf-8"
            ) as f:

                text = f.read()

        except Exception:
            continue


        if TARGET not in text:
            continue


        # ----------------------------------------------------
        # DETERMINE WHETHER THIS FILE CONTAINS THE DEFINITION
        # OR ONLY A REFERENCE TO THE FUNCTION.
        # ----------------------------------------------------

        definition_pattern = re.compile(
            r"def\s+_extract_socrata_rows\s*\("
        )

        contains_definition = bool(
            definition_pattern.search(text)
        )


        reference_count = text.count(
            TARGET
        )


        matches.append({
            "path": path,
            "definition": contains_definition,
            "references": reference_count,
        })


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print("=" * 78)
print("VLCP VERIFIED HELPER SOURCE SEARCH")
print("=" * 78)

print(
    "Target:",
    TARGET
)

print(
    "Files containing target:",
    len(matches)
)

print()


if matches:

    for item in matches:

        marker = (
            "✓ DEFINITION FOUND"
            if item["definition"]
            else "• REFERENCE ONLY"
        )

        print(marker)

        print(
            "  File:",
            item["path"]
        )

        print(
            "  Occurrences:",
            item["references"]
        )

        print()

else:

    print(
        "No persistent artifact contains "
        "the helper name."
    )


definitions = [
    item
    for item in matches
    if item["definition"]
]


print("-" * 78)

print(
    "Exact definitions found:",
    len(definitions)
)


if definitions:

    print(
        "\n✓ VERIFIED HELPER SOURCE "
        "IS AVAILABLE FOR RECOVERY"
    )

else:

    print(
        "\n⚠ EXACT HELPER DEFINITION "
        "NOT FOUND IN PERSISTENT ARTIFACTS"
    )


print("=" * 78)

VLCP VERIFIED HELPER SOURCE SEARCH
Target: _extract_socrata_rows
Files containing target: 3

• REFERENCE ONLY
  File: /content/drive/MyDrive/VLCP_PoC/verified_checkpoint_v1/vlcp_complete_verified_source.py
  Occurrences: 1

• REFERENCE ONLY
  File: /content/drive/MyDrive/VLCP_PoC/vlcp/core.py
  Occurrences: 1

• REFERENCE ONLY
  File: /content/drive/MyDrive/VLCP_PoC/vlcp/core_before_repair_001.py
  Occurrences: 1

------------------------------------------------------------------------------
Exact definitions found: 0

⚠ EXACT HELPER DEFINITION NOT FOUND IN PERSISTENT ARTIFACTS


In [6]:
# ============================================================
# VLCP — PACKAGE REPAIR 002
#
# Restores the exact verified implementation of:
#     _extract_socrata_rows
#
# Source:
#   Original verified development runtime
#   recovered via inspect.getsource()
#
# No business logic is reconstructed or rewritten.
# ============================================================

import ast
import os
import shutil


CORE_FILE = (
    "/content/drive/MyDrive/"
    "VLCP_PoC/vlcp/core.py"
)

HELPER_FILE = (
    "/content/drive/MyDrive/"
    "VLCP_PoC/recovered_verified_helpers/"
    "_extract_socrata_rows.py"
)

BACKUP_FILE = (
    "/content/drive/MyDrive/"
    "VLCP_PoC/vlcp/core_before_repair_002.py"
)


# ------------------------------------------------------------
# 1. VERIFY FILES
# ------------------------------------------------------------

if not os.path.isfile(CORE_FILE):

    raise FileNotFoundError(
        f"VLCP core not found: {CORE_FILE}"
    )


if not os.path.isfile(HELPER_FILE):

    raise FileNotFoundError(
        f"Recovered helper not found: {HELPER_FILE}"
    )


# ------------------------------------------------------------
# 2. READ PACKAGE + VERIFIED HELPER
# ------------------------------------------------------------

with open(
    CORE_FILE,
    "r",
    encoding="utf-8"
) as f:

    core_source = f.read()


with open(
    HELPER_FILE,
    "r",
    encoding="utf-8"
) as f:

    helper_source = f.read()


# ------------------------------------------------------------
# 3. VALIDATE RECOVERED HELPER
# ------------------------------------------------------------

helper_tree = ast.parse(
    helper_source
)


helper_defs = [
    node
    for node in helper_tree.body
    if isinstance(
        node,
        ast.FunctionDef
    )
]


if len(helper_defs) != 1:

    raise RuntimeError(
        "Recovered helper file must contain "
        "exactly one function definition."
    )


if (
    helper_defs[0].name
    != "_extract_socrata_rows"
):

    raise RuntimeError(
        "Recovered helper contains unexpected "
        f"function: {helper_defs[0].name}"
    )


compile(
    helper_source,
    HELPER_FILE,
    "exec"
)


# ------------------------------------------------------------
# 4. MAKE BACKUP BEFORE PACKAGE MODIFICATION
# ------------------------------------------------------------

if not os.path.exists(
    BACKUP_FILE
):

    shutil.copy2(
        CORE_FILE,
        BACKUP_FILE
    )


# ------------------------------------------------------------
# 5. CHECK WHETHER DEFINITION ALREADY EXISTS
# ------------------------------------------------------------

core_tree = ast.parse(
    core_source
)


existing_functions = {
    node.name
    for node in core_tree.body
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef
        )
    )
}


if (
    "_extract_socrata_rows"
    in existing_functions
):

    print(
        "_extract_socrata_rows already "
        "exists in package."
    )

else:

    # --------------------------------------------------------
    # Insert immediately before _socrata_query because this
    # helper belongs to the Socrata retrieval layer.
    # --------------------------------------------------------

    marker = (
        "def _socrata_query("
    )


    marker_position = (
        core_source.find(
            marker
        )
    )


    if marker_position == -1:

        raise RuntimeError(
            "Could not locate _socrata_query "
            "in core.py."
        )


    repaired_source = (
        core_source[:marker_position]
        + helper_source.rstrip()
        + "\n\n\n"
        + core_source[marker_position:]
    )


    with open(
        CORE_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            repaired_source
        )


# ------------------------------------------------------------
# 6. VALIDATE COMPLETE PACKAGE
# ------------------------------------------------------------

with open(
    CORE_FILE,
    "r",
    encoding="utf-8"
) as f:

    final_source = f.read()


compile(
    final_source,
    CORE_FILE,
    "exec"
)


final_tree = ast.parse(
    final_source
)


final_functions = {
    node.name
    for node in final_tree.body
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef
        )
    )
}


assert (
    "_extract_socrata_rows"
    in final_functions
), (
    "_extract_socrata_rows was not "
    "persisted into the package."
)


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print("=" * 78)
print("VLCP PACKAGE REPAIR 002")
print("=" * 78)

print(
    "Recovered helper:",
    HELPER_FILE
)

print(
    "Persistent package:",
    CORE_FILE
)

print(
    "Backup:",
    BACKUP_FILE
)

print(
    "Helper source:",
    "EXACT VERIFIED RUNTIME SOURCE"
)

print(
    "Helper reconstructed:",
    False
)

print(
    "Package compile test:",
    "PASS"
)

print(
    "Function definitions:",
    len(final_functions)
)

print()

print(
    "✓ _extract_socrata_rows "
    "RESTORED TO VLCP PACKAGE"
)

print("=" * 78)

VLCP PACKAGE REPAIR 002
Recovered helper: /content/drive/MyDrive/VLCP_PoC/recovered_verified_helpers/_extract_socrata_rows.py
Persistent package: /content/drive/MyDrive/VLCP_PoC/vlcp/core.py
Backup: /content/drive/MyDrive/VLCP_PoC/vlcp/core_before_repair_002.py
Helper source: EXACT VERIFIED RUNTIME SOURCE
Helper reconstructed: False
Package compile test: PASS
Function definitions: 34

✓ _extract_socrata_rows RESTORED TO VLCP PACKAGE


In [7]:
# ============================================================
# VLCP — PACKAGE REPAIR 002
#
# Restores the exact verified implementation of:
#     _extract_socrata_rows
#
# Source:
#   Original verified development runtime
#   recovered via inspect.getsource()
#
# No business logic is reconstructed or rewritten.
# ============================================================

import ast
import os
import shutil


CORE_FILE = (
    "/content/drive/MyDrive/"
    "VLCP_PoC/vlcp/core.py"
)

HELPER_FILE = (
    "/content/drive/MyDrive/"
    "VLCP_PoC/recovered_verified_helpers/"
    "_extract_socrata_rows.py"
)

BACKUP_FILE = (
    "/content/drive/MyDrive/"
    "VLCP_PoC/vlcp/core_before_repair_002.py"
)


# ------------------------------------------------------------
# 1. VERIFY FILES
# ------------------------------------------------------------

if not os.path.isfile(CORE_FILE):

    raise FileNotFoundError(
        f"VLCP core not found: {CORE_FILE}"
    )


if not os.path.isfile(HELPER_FILE):

    raise FileNotFoundError(
        f"Recovered helper not found: {HELPER_FILE}"
    )


# ------------------------------------------------------------
# 2. READ PACKAGE + VERIFIED HELPER
# ------------------------------------------------------------

with open(
    CORE_FILE,
    "r",
    encoding="utf-8"
) as f:

    core_source = f.read()


with open(
    HELPER_FILE,
    "r",
    encoding="utf-8"
) as f:

    helper_source = f.read()


# ------------------------------------------------------------
# 3. VALIDATE RECOVERED HELPER
# ------------------------------------------------------------

helper_tree = ast.parse(
    helper_source
)


helper_defs = [
    node
    for node in helper_tree.body
    if isinstance(
        node,
        ast.FunctionDef
    )
]


if len(helper_defs) != 1:

    raise RuntimeError(
        "Recovered helper file must contain "
        "exactly one function definition."
    )


if (
    helper_defs[0].name
    != "_extract_socrata_rows"
):

    raise RuntimeError(
        "Recovered helper contains unexpected "
        f"function: {helper_defs[0].name}"
    )


compile(
    helper_source,
    HELPER_FILE,
    "exec"
)


# ------------------------------------------------------------
# 4. MAKE BACKUP BEFORE PACKAGE MODIFICATION
# ------------------------------------------------------------

if not os.path.exists(
    BACKUP_FILE
):

    shutil.copy2(
        CORE_FILE,
        BACKUP_FILE
    )


# ------------------------------------------------------------
# 5. CHECK WHETHER DEFINITION ALREADY EXISTS
# ------------------------------------------------------------

core_tree = ast.parse(
    core_source
)


existing_functions = {
    node.name
    for node in core_tree.body
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef
        )
    )
}


if (
    "_extract_socrata_rows"
    in existing_functions
):

    print(
        "_extract_socrata_rows already "
        "exists in package."
    )

else:

    # --------------------------------------------------------
    # Insert immediately before _socrata_query because this
    # helper belongs to the Socrata retrieval layer.
    # --------------------------------------------------------

    marker = (
        "def _socrata_query("
    )


    marker_position = (
        core_source.find(
            marker
        )
    )


    if marker_position == -1:

        raise RuntimeError(
            "Could not locate _socrata_query "
            "in core.py."
        )


    repaired_source = (
        core_source[:marker_position]
        + helper_source.rstrip()
        + "\n\n\n"
        + core_source[marker_position:]
    )


    with open(
        CORE_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            repaired_source
        )


# ------------------------------------------------------------
# 6. VALIDATE COMPLETE PACKAGE
# ------------------------------------------------------------

with open(
    CORE_FILE,
    "r",
    encoding="utf-8"
) as f:

    final_source = f.read()


compile(
    final_source,
    CORE_FILE,
    "exec"
)


final_tree = ast.parse(
    final_source
)


final_functions = {
    node.name
    for node in final_tree.body
    if isinstance(
        node,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef
        )
    )
}


assert (
    "_extract_socrata_rows"
    in final_functions
), (
    "_extract_socrata_rows was not "
    "persisted into the package."
)


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print("=" * 78)
print("VLCP PACKAGE REPAIR 002")
print("=" * 78)

print(
    "Recovered helper:",
    HELPER_FILE
)

print(
    "Persistent package:",
    CORE_FILE
)

print(
    "Backup:",
    BACKUP_FILE
)

print(
    "Helper source:",
    "EXACT VERIFIED RUNTIME SOURCE"
)

print(
    "Helper reconstructed:",
    False
)

print(
    "Package compile test:",
    "PASS"
)

print(
    "Function definitions:",
    len(final_functions)
)

print()

print(
    "✓ _extract_socrata_rows "
    "RESTORED TO VLCP PACKAGE"
)

print("=" * 78)

_extract_socrata_rows already exists in package.
VLCP PACKAGE REPAIR 002
Recovered helper: /content/drive/MyDrive/VLCP_PoC/recovered_verified_helpers/_extract_socrata_rows.py
Persistent package: /content/drive/MyDrive/VLCP_PoC/vlcp/core.py
Backup: /content/drive/MyDrive/VLCP_PoC/vlcp/core_before_repair_002.py
Helper source: EXACT VERIFIED RUNTIME SOURCE
Helper reconstructed: False
Package compile test: PASS
Function definitions: 34

✓ _extract_socrata_rows RESTORED TO VLCP PACKAGE


In [8]:
# ============================================================
# VLCP — FINAL STATIC DEPENDENCY AUDIT
# ============================================================

import ast
import builtins

CORE_FILE = (
    "/content/drive/MyDrive/"
    "VLCP_PoC/vlcp/core.py"
)

with open(CORE_FILE, "r", encoding="utf-8") as f:
    source = f.read()

tree = ast.parse(source)

# ------------------------------------------------------------
# COLLECT ALL TOP-LEVEL NAMES PROVIDED BY THE MODULE
# ------------------------------------------------------------

top_level_names = set()

for node in tree.body:

    if isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)
    ):
        top_level_names.add(node.name)

    elif isinstance(node, ast.Import):

        for alias in node.names:
            top_level_names.add(
                alias.asname or alias.name.split(".")[0]
            )

    elif isinstance(node, ast.ImportFrom):

        for alias in node.names:
            top_level_names.add(
                alias.asname or alias.name
            )

    elif isinstance(node, (ast.Assign, ast.AnnAssign)):

        targets = (
            node.targets
            if isinstance(node, ast.Assign)
            else [node.target]
        )

        for target in targets:
            if isinstance(target, ast.Name):
                top_level_names.add(target.id)


# ------------------------------------------------------------
# FIND GLOBAL REFERENCES USED INSIDE FUNCTIONS
# ------------------------------------------------------------

builtins_set = set(dir(builtins))
unresolved = set()


for function in [
    node
    for node in tree.body
    if isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef)
    )
]:

    local_names = set()

    # Parameters
    for arg in (
        list(function.args.posonlyargs)
        + list(function.args.args)
        + list(function.args.kwonlyargs)
    ):
        local_names.add(arg.arg)

    if function.args.vararg:
        local_names.add(
            function.args.vararg.arg
        )

    if function.args.kwarg:
        local_names.add(
            function.args.kwarg.arg
        )

    # Assignments and exception aliases
    for node in ast.walk(function):

        if isinstance(node, ast.Name):

            if isinstance(
                node.ctx,
                (ast.Store, ast.Param)
            ):
                local_names.add(node.id)

        elif isinstance(
            node,
            ast.ExceptHandler
        ):

            if isinstance(node.name, str):
                local_names.add(node.name)

    # Loaded names
    for node in ast.walk(function):

        if not isinstance(node, ast.Name):
            continue

        if not isinstance(node.ctx, ast.Load):
            continue

        name = node.id

        if name in local_names:
            continue

        if name in builtins_set:
            continue

        if name in top_level_names:
            continue

        unresolved.add(name)


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

function_count = sum(
    isinstance(
        node,
        (ast.FunctionDef, ast.AsyncFunctionDef)
    )
    for node in tree.body
)

print("=" * 78)
print("VLCP — FINAL STATIC DEPENDENCY AUDIT")
print("=" * 78)

print(
    "Function definitions:",
    function_count
)

print(
    "Top-level definitions:",
    len(top_level_names)
)

print(
    "Unresolved references:",
    len(unresolved)
)


if unresolved:

    print("\nUNRESOLVED REFERENCES")
    print("-" * 78)

    for name in sorted(unresolved):
        print("✗", name)

    print("\n⚠ PACKAGE DEPENDENCY AUDIT FAILED")

else:

    print(
        "\n✓ NO UNRESOLVED GLOBAL REFERENCES"
    )

    print(
        "✓ VLCP PACKAGE STATIC "
        "DEPENDENCY AUDIT PASSED"
    )


print("=" * 78)

VLCP — FINAL STATIC DEPENDENCY AUDIT
Function definitions: 34
Top-level definitions: 53
Unresolved references: 0

✓ NO UNRESOLVED GLOBAL REFERENCES
✓ VLCP PACKAGE STATIC DEPENDENCY AUDIT PASSED


In [10]:
# ============================================================
# VLCP — EQUIPMENT RETURN CONTRACT INSPECTION
#
# No API calls.
# No package modification.
#
# Purpose:
# Determine the exact return structure of the already-successful
# run_vlcp_operational_equipment() execution.
# ============================================================

print("=" * 78)
print("VLCP — EQUIPMENT RETURN CONTRACT")
print("=" * 78)

print(
    "Result type:",
    type(equipment_result).__name__
)

print(
    "Top-level keys:",
    len(equipment_result)
)

print()


for key in sorted(
    equipment_result.keys()
):

    value = equipment_result[key]

    print(
        f"{key:<35} "
        f"{type(value).__name__:<15}",
        end=""
    )

    # Show safe scalar values.
    if isinstance(
        value,
        (str, int, float, bool, type(None))
    ):

        print(
            repr(value)
        )

    # For containers, show size without dumping
    # potentially large evidence payloads.
    elif isinstance(
        value,
        (list, tuple, set, dict)
    ):

        print(
            f"len={len(value)}"
        )

    else:

        print()


print()
print("-" * 78)


# ------------------------------------------------------------
# LOOK INSIDE COMMON NESTED RESULT SECTIONS
# ------------------------------------------------------------

for section_name in [
    "counts",
    "evidence",
    "equipment",
    "summary",
]:

    section = equipment_result.get(
        section_name
    )

    if isinstance(
        section,
        dict
    ):

        print(
            f"\n{section_name.upper()} KEYS"
        )

        print("-" * 78)

        for key in sorted(
            section.keys()
        ):

            value = section[key]

            if isinstance(
                value,
                (str, int, float, bool, type(None))
            ):

                print(
                    f"{key:<35} {value!r}"
                )

            elif isinstance(
                value,
                (list, tuple, set, dict)
            ):

                print(
                    f"{key:<35} "
                    f"{type(value).__name__} "
                    f"len={len(value)}"
                )

            else:

                print(
                    f"{key:<35} "
                    f"{type(value).__name__}"
                )


print()
print("=" * 78)

print(
    "✓ RETURN CONTRACT INSPECTION COMPLETE"
)

print("=" * 78)

VLCP — EQUIPMENT RETURN CONTRACT
Result type: dict
Top-level keys: 9

company_name                        str            'US SERVICES LLC'
equipment_index                     dict           len=201
equipment_observation_count         int            263
inspection_count                    int            142
likely_unique_equipment             int            201
sources                             list           len=2
usdot                               str            '951224'
vin_decode_index                    dict           len=10
vin_identified_assets               int            10

------------------------------------------------------------------------------

✓ RETURN CONTRACT INSPECTION COMPLETE


In [11]:
# ============================================================
# VLCP — EQUIPMENT PORTABILITY REGRESSION
# FINAL CONTRACT VALIDATION
#
# Uses the EXISTING equipment_result.
# No FMCSA/NHTSA calls.
# No package modifications.
# ============================================================


# ------------------------------------------------------------
# ACTUAL FROZEN RETURN CONTRACT
# ------------------------------------------------------------

required_keys = {
    "company_name",
    "usdot",
    "inspection_count",
    "equipment_observation_count",
    "likely_unique_equipment",
    "vin_identified_assets",
    "equipment_index",
    "vin_decode_index",
    "sources",
}


missing_keys = (
    required_keys
    - set(equipment_result.keys())
)


assert not missing_keys, (
    "Missing equipment contract keys: "
    f"{sorted(missing_keys)}"
)


# ------------------------------------------------------------
# VALUES
# ------------------------------------------------------------

company_name = (
    equipment_result["company_name"]
)

usdot = str(
    equipment_result["usdot"]
)

inspection_count = (
    equipment_result["inspection_count"]
)

observation_count = (
    equipment_result[
        "equipment_observation_count"
    ]
)

unique_observed_count = (
    equipment_result[
        "likely_unique_equipment"
    ]
)

vin_identified_assets = (
    equipment_result[
        "vin_identified_assets"
    ]
)

equipment_index = (
    equipment_result[
        "equipment_index"
    ]
)

vin_decode_index = (
    equipment_result[
        "vin_decode_index"
    ]
)


# ------------------------------------------------------------
# STRUCTURAL REGRESSION ASSERTIONS
#
# We intentionally avoid exact live-source counts.
# ------------------------------------------------------------

assert company_name, (
    "Carrier identity missing."
)

assert usdot == "951224", (
    f"Unexpected USDOT: {usdot}"
)

assert inspection_count > 0, (
    "No FMCSA inspections returned."
)

assert observation_count > 0, (
    "No equipment observations returned."
)

assert unique_observed_count > 0, (
    "No unique equipment observations returned."
)

assert (
    observation_count
    >= unique_observed_count
), (
    "Unique observed equipment cannot exceed "
    "total equipment observations."
)

assert (
    len(equipment_index)
    == unique_observed_count
), (
    "Equipment index size does not match "
    "unique observed equipment count."
)

assert vin_identified_assets > 0, (
    "No VIN-identified assets returned."
)

assert (
    len(vin_decode_index)
    == vin_identified_assets
), (
    "VIN decode index size does not match "
    "VIN-identified asset count."
)

assert (
    vin_identified_assets <= VIN_LIMIT
), (
    "VIN result exceeded requested decode limit."
)


# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------

print("=" * 78)
print("VLCP — EQUIPMENT PORTABILITY REGRESSION")
print("=" * 78)

print(
    "Carrier:",
    company_name
)

print(
    "USDOT:",
    usdot
)

print(
    "FMCSA inspections:",
    inspection_count
)

print(
    "Equipment observations:",
    observation_count
)

print(
    "Unique equipment observed:",
    unique_observed_count
)

print(
    "VIN-identified assets:",
    vin_identified_assets
)

print(
    "VIN decode records:",
    len(vin_decode_index)
)

print(
    "Missing contract keys:",
    len(missing_keys)
)

print()

print(
    "✓ VLCP PORTABLE EQUIPMENT v1 "
    "PASSED FRESH-NOTEBOOK TEST"
)

print("=" * 78)

VLCP — EQUIPMENT PORTABILITY REGRESSION
Carrier: US SERVICES LLC
USDOT: 951224
FMCSA inspections: 142
Equipment observations: 263
Unique equipment observed: 201
VIN-identified assets: 10
VIN decode records: 10
Missing contract keys: 0

✓ VLCP PORTABLE EQUIPMENT v1 PASSED FRESH-NOTEBOOK TEST


In [12]:
# ============================================================
# VLCP — OPERATIONAL ENTRY POINT v1
#
# Executes the complete frozen VLCP verification workflow
# through ONE public function.
#
# No policy decisions.
# No broker qualification.
# No PASS/FAIL.
#
# FACT -> EVIDENCE only.
# ============================================================

import importlib
from google.colab import userdata

import vlcp.core as core


# ------------------------------------------------------------
# LOAD CURRENT PERSISTENT PACKAGE
# ------------------------------------------------------------

core = importlib.reload(core)

core.FMCSA_WEBKEY = userdata.get(
    "FMCSA_WEBKEY"
)

if not core.FMCSA_WEBKEY:
    raise RuntimeError(
        "FMCSA_WEBKEY not available in Colab Secrets."
    )


# ------------------------------------------------------------
# DYNAMIC INPUT
# ------------------------------------------------------------

TEST_USDOT = "951224"


# ------------------------------------------------------------
# SINGLE VLCP OPERATIONAL ENTRY POINT
# ------------------------------------------------------------

result = core.run_vlcp_verification(
    TEST_USDOT,
    include_equipment=True,
    vin_limit=10,
    decode_delay=0.10
)


# ------------------------------------------------------------
# INSPECT OPERATIONAL CONTRACT
# ------------------------------------------------------------

print()
print("=" * 78)
print("VLCP — OPERATIONAL VERIFICATION CONTRACT")
print("=" * 78)

print(
    "Result type:",
    type(result).__name__
)

print(
    "Top-level keys:",
    len(result)
)

print()


for key in sorted(result.keys()):

    value = result[key]

    if isinstance(
        value,
        (str, int, float, bool, type(None))
    ):

        description = repr(value)

    elif isinstance(
        value,
        (dict, list, tuple, set)
    ):

        description = (
            f"{type(value).__name__} "
            f"len={len(value)}"
        )

    else:

        description = (
            type(value).__name__
        )

    print(
        f"{key:<30} {description}"
    )


# ------------------------------------------------------------
# MODULE STATUS
# ------------------------------------------------------------

print()
print("MODULE STATUS")
print("-" * 78)

module_status = (
    result.get("module_status")
    or {}
)

for module, status in module_status.items():

    print(
        f"{module:<24} {status}"
    )


# ------------------------------------------------------------
# ERRORS
# ------------------------------------------------------------

print()
print("ERRORS")
print("-" * 78)

errors = result.get(
    "errors"
) or []

if errors:

    for error in errors:
        print("•", error)

else:

    print("None")


# ------------------------------------------------------------
# MINIMUM OPERATIONAL CONTRACT
# ------------------------------------------------------------

required_sections = {
    "usdot",
    "identity",
    "authority",
    "insurance",
    "insurance_lifecycle",
    "equipment",
    "module_status",
    "errors",
}


missing_sections = (
    required_sections
    - set(result.keys())
)


assert not missing_sections, (
    "Missing operational sections: "
    f"{sorted(missing_sections)}"
)


assert (
    str(result["usdot"])
    == TEST_USDOT
)


assert (
    module_status.get("identity")
    == "FOUND"
)


assert (
    module_status.get("authority")
    == "RETRIEVED"
)


assert (
    module_status.get("insurance")
    == "RETRIEVED"
)


assert (
    module_status.get(
        "insurance_lifecycle"
    )
    == "RETRIEVED"
)


assert (
    module_status.get("equipment")
    == "RETRIEVED"
)


assert not errors


print()
print("=" * 78)

print(
    "✓ VLCP OPERATIONAL ENTRY POINT v1 PASSED"
)

print(
    "✓ ONE USDOT -> COMPLETE VERIFIED EVIDENCE PROFILE"
)

print("=" * 78)

VLCP AUTHORITATIVE VIN DECODING
Candidate VINs: 10
Source: NHTSA vPIC
----------------------------------------------------------------------
[1/10] 1AJC40260T1002016 -> DECODED
[2/10] 1AJC40265T1002769 -> DECODED
[3/10] 1DW1C452X1E499580 -> DECODED
[4/10] 1FUGGHDV9LLLT8409 -> DECODED
[5/10] 1FUJA6CGX3LH34804 -> DECODED
[6/10] 1FUJA6CK28DY52591 -> DECODED
[7/10] 1FUJA6CK36LV99316 -> DECODED
[8/10] 1FUJA6CK47LX77848 -> DECODED
[9/10] 1FUJA6CK57LW40286 -> DECODED
[10/10] 1FUJA6CK65LN64042 -> DECODED
----------------------------------------------------------------------
Decode summary:
  DECODED: 10

VLCP Verified Equipment Summary

Carrier: US SERVICES LLC
USDOT: 951224

Equipment | Year | Make               | Model              | VIN               | Verification
----------+------+--------------------+--------------------+-------------------+-------------
Tractor   | 2003 | Freightliner       | Columbia           | 1FUJA6CGX3LH34804 | Verified    
Tractor   | 2005 | Freightliner       | C

In [13]:
# ============================================================
# VLCP — CREATE VERIFIED REDUNDANCY RELEASE
#
# Creates a clean ZIP from the EXACT package that passed:
#   - fresh-notebook core verification
#   - static dependency audit
#   - equipment portability regression
#
# No credentials included.
# No source reconstruction.
# ============================================================

import os
import json
import shutil
import hashlib
import zipfile
from datetime import datetime, UTC


# ------------------------------------------------------------
# SOURCE — EXACT VERIFIED PACKAGE
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/drive/MyDrive/VLCP_PoC"
)

SOURCE_PACKAGE = os.path.join(
    PROJECT_ROOT,
    "vlcp"
)

CORE_FILE = os.path.join(
    SOURCE_PACKAGE,
    "core.py"
)

INIT_FILE = os.path.join(
    SOURCE_PACKAGE,
    "__init__.py"
)


# ------------------------------------------------------------
# REDUNDANCY DESTINATION
# ------------------------------------------------------------

REDUNDANCY_ROOT = os.path.join(
    PROJECT_ROOT,
    "redundancy"
)

RELEASE_NAME = (
    "VLCP_Portable_Verification_v1_VERIFIED"
)

RELEASE_DIR = os.path.join(
    REDUNDANCY_ROOT,
    RELEASE_NAME
)

PACKAGE_DEST = os.path.join(
    RELEASE_DIR,
    "vlcp"
)

ZIP_FILE = os.path.join(
    REDUNDANCY_ROOT,
    RELEASE_NAME + ".zip"
)


# ------------------------------------------------------------
# VERIFY OPERATIONAL SOURCE EXISTS
# ------------------------------------------------------------

for required in [
    CORE_FILE,
    INIT_FILE,
]:

    if not os.path.isfile(required):

        raise FileNotFoundError(
            f"Required verified file missing: {required}"
        )


# ------------------------------------------------------------
# CREATE CLEAN RELEASE DIRECTORY
# ------------------------------------------------------------

os.makedirs(
    REDUNDANCY_ROOT,
    exist_ok=True
)


if os.path.exists(
    RELEASE_DIR
):

    shutil.rmtree(
        RELEASE_DIR
    )


os.makedirs(
    RELEASE_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# COPY EXACT VERIFIED PACKAGE
# ------------------------------------------------------------

shutil.copytree(
    SOURCE_PACKAGE,
    PACKAGE_DEST,
    ignore=shutil.ignore_patterns(
        "__pycache__",
        "*.pyc"
    )
)


# ------------------------------------------------------------
# SHA-256 CHECKSUM
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


source_core_hash = sha256_file(
    CORE_FILE
)

backup_core_hash = sha256_file(
    os.path.join(
        PACKAGE_DEST,
        "core.py"
    )
)


assert (
    source_core_hash
    == backup_core_hash
), (
    "Redundancy core.py does not match "
    "verified operational core.py."
)


# ------------------------------------------------------------
# VERIFY PACKAGE COMPILES
# ------------------------------------------------------------

with open(
    os.path.join(
        PACKAGE_DEST,
        "core.py"
    ),
    "r",
    encoding="utf-8"
) as f:

    core_source = f.read()


compile(
    core_source,
    "vlcp/core.py",
    "exec"
)


# ------------------------------------------------------------
# RELEASE MANIFEST
# ------------------------------------------------------------

manifest = {

    "project":
        "Verified Logistics Chain Protocol",

    "release":
        "VLCP Portable Verification v1",

    "status":
        "VERIFIED_OPERATIONAL_REDUNDANCY",

    "created_at_utc":
        datetime.now(
            UTC
        ).isoformat(),

    "source_reconstructed":
        False,

    "exact_verified_source":
        True,

    "credential_included":
        False,

    "credential_requirement":
        "FMCSA_WEBKEY",

    "credential_storage":
        "External / Colab Secrets",

    "design_boundary":
        "FACT -> EVIDENCE -> POLICY -> DECISION",

    "verified_components": {

        "carrier_identity":
            "PASS",

        "authority":
            "PASS",

        "insurance":
            "PASS",

        "insurance_lifecycle":
            "PASS",

        "equipment":
            "PASS",

        "nhtsa_vin_decode":
            "PASS",

        "fresh_notebook_core":
            "PASS",

        "fresh_notebook_equipment":
            "PASS",

        "static_dependency_audit":
            "PASS"
    },

    "static_unresolved_references":
        0,

    "function_definitions":
        34,

    "known_packaging_repairs": [

        "SOCRATA_DOMAIN persisted",

        "_extract_socrata_rows recovered "
        "from exact verified runtime source"
    ],

    "latest_equipment_regression": {

        "usdot":
            "951224",

        "carrier":
            "US SERVICES LLC",

        "inspection_count":
            142,

        "equipment_observation_count":
            263,

        "unique_equipment_observed":
            201,

        "vin_identified_assets":
            10,

        "vin_decode_records":
            10
    },

    "sha256": {

        "vlcp/core.py":
            source_core_hash
    }
}


MANIFEST_FILE = os.path.join(
    RELEASE_DIR,
    "manifest.json"
)


with open(
    MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ------------------------------------------------------------
# README
# ------------------------------------------------------------

README = """VLCP PORTABLE VERIFICATION v1
================================

STATUS
------
VERIFIED / OPERATIONAL REDUNDANCY

This release contains the exact VLCP package that passed the
fresh-notebook portability and equipment regression tests.

PUBLIC ENTRY POINTS
-------------------
verify_carrier(usdot)

run_vlcp_verification(
    usdot,
    include_equipment=True,
    vin_limit=10,
    decode_delay=0.10
)

SECURITY
--------
FMCSA_WEBKEY is NOT stored in this release.

The credential must be supplied externally, such as through
Google Colab Secrets.

DESIGN BOUNDARY
---------------
FACT -> EVIDENCE -> POLICY -> DECISION

VLCP verification establishes evidence.

Carrier qualification, broker policy, shipper policy and
selection decisions belong to separate downstream layers.

VERIFIED MODULES
----------------
Carrier Identity
Authority
Insurance
Insurance Lifecycle
FMCSA Inspection / Equipment Evidence
NHTSA VIN Decoding
Operational Orchestration

PORTABILITY
-----------
Fresh notebook core test: PASS
Fresh notebook equipment test: PASS
Static unresolved dependencies: 0

IMPORTANT TERMINOLOGY
---------------------
The internal legacy key:

    likely_unique_equipment

is retained for backward compatibility.

Production reporting should display this evidence as:

    Unique equipment observed

because historical FMCSA inspection observations do not prove
current ownership or current fleet size.
"""


with open(
    os.path.join(
        RELEASE_DIR,
        "README.txt"
    ),
    "w",
    encoding="utf-8"
) as f:

    f.write(
        README
    )


# ------------------------------------------------------------
# CREATE ZIP
# ------------------------------------------------------------

if os.path.exists(
    ZIP_FILE
):

    os.remove(
        ZIP_FILE
    )


with zipfile.ZipFile(
    ZIP_FILE,
    "w",
    zipfile.ZIP_DEFLATED
) as archive:

    for root, dirs, files in os.walk(
        RELEASE_DIR
    ):

        dirs[:] = [
            d for d in dirs
            if d != "__pycache__"
        ]

        for filename in files:

            if filename.endswith(
                ".pyc"
            ):
                continue

            full_path = os.path.join(
                root,
                filename
            )

            archive_name = os.path.relpath(
                full_path,
                RELEASE_DIR
            )

            archive.write(
                full_path,
                archive_name
            )


# ------------------------------------------------------------
# VERIFY ZIP
# ------------------------------------------------------------

with zipfile.ZipFile(
    ZIP_FILE,
    "r"
) as archive:

    bad_file = archive.testzip()

    zip_contents = (
        archive.namelist()
    )


assert bad_file is None, (
    f"ZIP integrity failure: {bad_file}"
)


assert "vlcp/core.py" in zip_contents
assert "vlcp/__init__.py" in zip_contents
assert "manifest.json" in zip_contents
assert "README.txt" in zip_contents


# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------

print("=" * 78)
print("VLCP — VERIFIED REDUNDANCY RELEASE")
print("=" * 78)

print(
    "Release:",
    RELEASE_NAME
)

print(
    "Source reconstructed:",
    False
)

print(
    "Exact verified core:",
    True
)

print(
    "Compile test:",
    "PASS"
)

print(
    "SHA-256 verified:",
    True
)

print(
    "ZIP integrity:",
    "PASS"
)

print(
    "Credential included:",
    False
)

print()

print(
    "Folder:",
    RELEASE_DIR
)

print(
    "ZIP:",
    ZIP_FILE
)

print()

print(
    "core.py SHA-256:"
)

print(
    source_core_hash
)

print()

print(
    "✓ VLCP VERIFIED OPERATIONAL "
    "REDUNDANCY CREATED"
)

print("=" * 78)

VLCP — VERIFIED REDUNDANCY RELEASE
Release: VLCP_Portable_Verification_v1_VERIFIED
Source reconstructed: False
Exact verified core: True
Compile test: PASS
SHA-256 verified: True
ZIP integrity: PASS
Credential included: False

Folder: /content/drive/MyDrive/VLCP_PoC/redundancy/VLCP_Portable_Verification_v1_VERIFIED
ZIP: /content/drive/MyDrive/VLCP_PoC/redundancy/VLCP_Portable_Verification_v1_VERIFIED.zip

core.py SHA-256:
650f6bea16cb47ddbdbff9d6e99e3036055ddbd905395024b9e4f752fce99b08

✓ VLCP VERIFIED OPERATIONAL REDUNDANCY CREATED
